# สร้างชุดข้อมูล CENTRAL

รวมผล harmonization จากทั้ง 4 ชุดข้อมูล แบ่ง split กลาง และบันทึกรายงานตรวจสอบ รันจากบนลงล่างหลังจบ harmonization ของทุกชุด


## 1. เตรียม path และ schema กลาง

หาโฟลเดอร์ data-pipeline และกำหนดคอลัมน์หลักกับชื่อคลาสที่ตารางทุกชุดต้องใช้


In [ ]:
from pathlib import Path

PIPELINE_DIR = next(
    candidate
    for ancestor in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    for candidate in (ancestor, ancestor / "data-pipeline")
    if (candidate / "CENTRAL" / "feature_contract.json").is_file()
)

PROJECT_ROOT = PIPELINE_DIR.parent

import hashlib
import numpy as np
import pandas as pd
import json
import os

NUMERIC = ["duration", "bytes_forward", "bytes_backward"]


CATEGORIES = {
    "protocol": ["tcp", "udp", "icmp", "other"],
    "service_group": ["dns", "ftp", "http", "https_ssl", "smtp", "ssh", "other"],
    "conn_state_group": ["established", "no-reply", "reset", "other"],
}


CORE = NUMERIC + list(CATEGORIES)


CLASS_NAMES = [
    "Benign / Normal",
    "Denial of Service (DoS)",
    "Reconnaissance & Probing",
    "Exploits & PrivEsc",
    "Credential Abuse (R2L)",
    "Malware, C2 & Botnet",
]


def require(condition, message):
    if not condition:
        raise ValueError(message)


QUALITY_FLAGS = ['invalid_required_value', 'bytes_packet_conflict', 'state_counter_conflict']


## 2. เตรียมตัวตรวจตารางและคุณภาพ

กำหนดวิธีอ่านตารางและตรวจ record_id, label และกฎคุณภาพ พร้อมฟังก์ชันอ่าน checksum และ raw manifest


In [ ]:
def normalize_boolean_metadata(metadata):
    columns=[c for c in metadata if c.startswith(('has_','invalid_')) or c in QUALITY_FLAGS+['quality_ok','service_identified','state_reclassified','dedup_flag','is_kddtest21_view']]
    for col in columns:
        values=metadata[col].astype('string').str.lower()
        if not values.isin(['true','false','1','0']).all():raise ValueError(f'Invalid boolean metadata: {col}')
        metadata[col]=values.map({'true':True,'false':False,'1':True,'0':False}).astype(bool)
    return metadata


def text(values):
    return values.astype('string').str.strip().str.lower().replace({'':'', '-':'-'})


def quality_flags(features, metadata):
    f=features
    invalid=pd.Series(False,index=f.index)
    for col in ['duration','bytes_forward','bytes_backward']:
        values=pd.to_numeric(f[col],errors='coerce')
        invalid |= values.isna() | ~np.isfinite(values) | values.lt(0)
    invalid |= f.protocol.isna() | f.protocol.astype('string').str.strip().isin(['','-'])
    packet_conflict=pd.Series(False,index=f.index)
    for direction in ['forward','backward']:
        col='packets_'+direction
        if col in f:
            values=pd.to_numeric(f[col],errors='coerce')
            packet_conflict |= values.eq(0) & f['bytes_'+direction].gt(0)
            invalid |= f[col].notna() & (~np.isfinite(values) | values.lt(0))
    raw_state=text(metadata.get('source_connection_state',pd.Series(pd.NA,index=f.index,dtype='string')))
    has_response=f.bytes_backward.gt(0)
    if 'packets_backward' in f:has_response |= f.packets_backward.gt(0)
    # REQ/INT can have response traffic. Literal no-reply/S0 with a response is
    # contradictory source evidence; do not silently erase that evidence.
    state_conflict=raw_state.isin(['no-reply','s0']) & has_response
    return pd.DataFrame({'invalid_required_value':invalid,
                         'bytes_packet_conflict':packet_conflict,
                         'state_counter_conflict':state_conflict},index=f.index)


DATASETS = ('NSL-KDD', 'UNSW-NB15', 'CIC-IDS2017', 'BoT-IoT')


TACTICS = ['Baseline Operations','TA0040: Impact','TA0043/TA0007: Recon & Discovery','TA0001/TA0004: Initial Access & PrivEsc','TA0006: Credential Access','TA0011/TA0008: C2 & Lateral Movement']


COMMON_MAPPING = {'normal':0,'benign':0,'dos':1,'dos/ddos':1,'ddos':1,'probe':2,'probe/recon':2,'reconnaissance':2,'analysis':2,'exploits':3,'exploit/u2r':3,'shellcode':3,'u2r':3,'r2l':4,'brute-force/r2l':4,'generic':5,'generic/fuzzer':5,'fuzzers':5,'backdoor':5,'backdoors':5,'botnet/backdoor':5,'worms':5}


SOURCE_MAPPING = {
    'NSL-KDD': {k:COMMON_MAPPING[k] for k in ['normal','dos','probe','r2l','u2r']},
    'UNSW-NB15': {k:COMMON_MAPPING[k] for k in ['normal','analysis','backdoor','backdoors','dos','exploits','fuzzers','generic','reconnaissance','shellcode','worms']},
    'CIC-IDS2017': {**COMMON_MAPPING, 'dos hulk':1,'dos goldeneye':1,'dos slowloris':1,'dos slowhttptest':1,'heartbleed':3,'portscan':2,'ftp-patator':4,'ssh-patator':4,'web attack - brute force':4,'web attack - xss':3,'web attack - sql injection':3,'web attack brute force':4,'web attack xss':3,'web attack sql injection':3,'infiltration':3,'bot':5},
    # Project-level category grouping, not per-record MITRE attribution.
    'BoT-IoT': {**COMMON_MAPPING,'theft':5,'data_theft':5,'bruteforce':4},
}


def digest(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1024*1024), b''):h.update(block)
    return h.hexdigest()


def write_json(path, value):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(json.dumps(value,indent=2,ensure_ascii=False,default=lambda x:x.item() if hasattr(x,'item') else str(x)),encoding='utf8')


def raw_manifest(dataset_dir):
    return [{'file':p.relative_to(Path(dataset_dir)/'raw').as_posix(),'bytes':p.stat().st_size,'sha256':digest(p)} for p in sorted((Path(dataset_dir)/'raw').rglob('*')) if p.is_file() and p.suffix.lower() in {'.csv','.gz','.txt'}]


def read_tables(folder):
    # Source labels can be numeric (UNSW) or text (NSL/CIC). Preserve lexical
    # identifiers consistently across source tables, concatenation and CSV reload.
    types={name:'string' for name in ['record_id','label_raw','label_source_category','source_attack_category','orig_row_id','split_group','source_dataset','source_split','source_protocol','source_connection_state','source_service','service_basis','connection_state_basis','duration_unit','quality_issues','capture_time']}
    f,l,m=tuple(pd.read_csv(Path(folder)/(t+'.csv'),low_memory=False,dtype=types) for t in ['features','labels','metadata'])
    return f,l,normalize_boolean_metadata(m)


def validate_tables(f,l,m,allow_quarantined=False):
    require(len(f)>0 and len(f)==len(l)==len(m),'Empty tables or row-count mismatch')
    for name,frame in [('features',f),('labels',l),('metadata',m)]:
        require('record_id' in frame and frame.record_id.notna().all() and frame.record_id.astype(str).str.strip().ne('').all() and frame.record_id.is_unique,f'{name}: missing/duplicate record_id')
    require(f.record_id.equals(l.record_id) and f.record_id.equals(m.record_id),'Record IDs/order mismatch')
    require(not set(f.columns).intersection({'label','label_raw','label_source_category','attack_category','attack_category_id','label_binary','mitre_tactic','is_attack','source_dataset','sample_weight','data_split','is_synthetic'}),'Target/metadata leakage in features')
    require(set(CORE).issubset(f.columns),'Missing core columns')
    computed=quality_flags(f,m)
    eligible=~computed.any(axis=1)
    if 'quality_ok' in m:
        require(m.quality_ok.notna().all() and m.quality_ok.eq(eligible).all(),'quality_ok disagrees with recomputed quality rules')
        for col in computed:require(col in m and m[col].eq(computed[col]).all(),f'{col}: quality flag mismatch')
    require(allow_quarantined or eligible.all(),'Invalid duration/bytes/protocol or cross-field conflicts: quarantined records are not allowed in model splits')
    checked=f.loc[eligible] if allow_quarantined else f
    for c in NUMERIC:
        z=pd.to_numeric(checked[c],errors='coerce')
        require(np.isfinite(z).all() and z.ge(0).all(),f'{c}: missing, infinite or negative input')
    for c,values in CATEGORIES.items():require(checked[c].notna().all() and checked[c].isin(values).all(),f'{c}: missing/unknown categories')
    response=checked.bytes_backward.gt(0)
    if 'packets_backward' in checked:response |= checked.packets_backward.gt(0)
    require(not (checked.conn_state_group.eq('no-reply') & response).any(),'no-reply contradicts backward traffic')
    for c in ['packets_forward','packets_backward','mean_packet_size','ttl_forward','ttl_backward','tcp_window_forward','tcp_window_backward']:
        if c in checked:
            z=checked[c].dropna()
            require(np.isfinite(z).all() and z.ge(0).all(),f'{c}: invalid measured extension')
    require('label_binary' in l and l.label_binary.isin([0,1]).all(),'Invalid binary labels')
    if 'attack_category_id' in l:
        require(l.attack_category_id.isin(range(6)).all(),'Invalid central classes')
        require(l.label_binary.eq(l.attack_category_id.ne(0).astype(int)).all(),'Binary/multiclass labels disagree')
        require(l.attack_category.eq(l.attack_category_id.map(dict(enumerate(CLASS_NAMES)))).all(),'Class name/ID mismatch')
    if 'sample_weight' in m:require(np.isfinite(m.sample_weight).all() and m.sample_weight.gt(0).all(),'Invalid sample weights')
    for flag in ['invalid_duration','invalid_bytes_forward','invalid_bytes_backward']:
        if flag in m:require(not m.loc[eligible,flag].astype(str).str.lower().isin(['true','1']).any(),f'{flag}: flagged invalid rows')


## 3. เตรียม mapping และตรวจที่มาของข้อมูล

กำหนดวิธีแปลง label เป็นคลาสกลาง ตรวจ raw, cleaned และ harmonized เทียบกับ dependency stamp และกำหนดวิธีถ่วงน้ำหนักตามชุดข้อมูล


In [ ]:
def map_labels(labels, metadata):
    result=labels.copy(); ids=pd.Series(index=labels.index,dtype='int64')
    for source,index in metadata.groupby('source_dataset').groups.items():
        require(source in SOURCE_MAPPING,f'Unsupported source {source}')
        values=labels.loc[index,'label_source_category'].astype('string').str.strip().str.lower().str.replace(r'\s+',' ',regex=True)
        mapped=values.map(SOURCE_MAPPING[source])
        require(mapped.notna().all(),f'Unmapped labels in {source}: {values[mapped.isna()].value_counts().to_dict()}')
        ids.loc[index]=mapped.astype(int)
    ids=ids.astype(int)
    require(labels.label_binary.eq(ids.ne(0).astype(int)).all(),'Mapped categories contradict original binary truth')
    result['attack_category_id']=ids;result['attack_category']=ids.map(dict(enumerate(CLASS_NAMES)));result['mitre_tactic']=ids.map(dict(enumerate(TACTICS)))
    return result


def code_signature(pipeline_dir):
    p=Path(pipeline_dir)
    files=sorted([*p.glob('*.py'),*p.glob('CENTRAL/*.json'),*p.glob('CENTRAL/*.ipynb'),*p.glob('sub-dataset/*/*.ipynb'),*p.glob('sub-dataset/*/*.py')])
    signature = {f.relative_to(p).as_posix():digest(f) for f in files}
    shared = PROJECT_ROOT / "webapp/Dashboard/backend/src/models/central_contract.py"
    signature["../webapp/Dashboard/backend/src/models/central_contract.py"] = digest(shared)
    return signature


def cleaned_fingerprints(dataset_dir):
    d=Path(dataset_dir)
    names={'NSL-KDD':['KDDTrain_clean.csv','KDDTest_clean.csv','KDDTest-21_clean.csv'],
           'UNSW-NB15':['UNSW_NB15_training-set.csv','UNSW_NB15_testing-set.csv'],
           'CIC-IDS2017':['CIC_IDS2017_training-set.csv','CIC_IDS2017_testing-set.csv'],
           'BoT-IoT':['BoT_IoT_training-set.csv','BoT_IoT_testing-set.csv']}[d.name]
    for name in names:require((d/'output/cleansing'/name).is_file(),f'Missing cleaned file: {d.name}/{name}')
    return {name:digest(d/'output/cleansing'/name) for name in names}


def verify_dataset(dataset_dir):
    d=Path(dataset_dir);path=d/'output/harmonized/dependencies.json'
    require(path.is_file(),f'Missing dependency stamp: {path}; rerun harmonization')
    stamp=json.loads(path.read_text(encoding='utf8'))
    require(stamp['raw']==raw_manifest(d),'Raw changed since harmonization')
    require(stamp['code']==code_signature(d.parent.parent),'Pipeline code/contract changed since harmonization')
    require(stamp.get('cleaned')==cleaned_fingerprints(d),'Cleaned artifacts changed since harmonization')
    origin=d/'raw/source_provenance.json'
    require(stamp.get('source_provenance')==(digest(origin) if origin.exists() else None),'Source provenance changed since harmonization')
    for name,h in stamp['tables'].items():require(digest(d/'output/harmonized'/f'{name}.csv')==h,f'Harmonized table changed: {name}')


def normalize_weights(metadata):
    result=metadata.copy()
    # Equal total contribution per source, with average row weight equal to 1.
    counts=result.groupby('source_dataset').record_id.transform('size')
    result['sample_weight']=len(result)/(result.source_dataset.nunique()*counts.astype(float))
    return result


def collect_sources(pipeline_dir, allow_unverified=False):
    p=Path(pipeline_dir);result={}
    for ds in DATASETS:
        d=p/'sub-dataset'/ds
        files=raw_manifest(d);require(files,f'Missing raw files: {ds}')
        path=d/'raw/source_provenance.json'
        if path.is_file():
            info=json.loads(path.read_text(encoding='utf8'))
            require(info.get('files')==files,f'Raw/source provenance mismatch: {ds}')
        else:
            info={'mode':'supplied_local' if ds in ['NSL-KDD','UNSW-NB15'] else 'unverified','files':files,'note':'Existing local raw files; origin not independently authenticated'}
            if ds=='CIC-IDS2017':
                first=pd.read_csv(d/'raw/CIC_IDS2017_training-set.csv',nrows=100)
                if 'id' in first and first.id.astype(str).str.startswith(('CIC_BENIGN_','CIC_DOS_','CIC_PROBE_')).any():info['mode']='emulate'
        if ds in ['CIC-IDS2017','BoT-IoT'] and info['mode']!='online':
            require(allow_unverified,f'{ds}: {info["mode"]} source. Supply documented online raw data or explicitly use --allow-unverified-data for a diagnostic run.')
        result[ds]=info
    return result


## 4. อ่านข้อมูลและคัดแถวที่ผ่านเกณฑ์

อ่านผล harmonization จากทั้ง 4 ชุด ตรวจ dependency stamp ตัดแถวที่ติดธงคุณภาพออกจากชุดที่จะใช้เทรน และแปลง label เป็นคลาสกลาง


In [ ]:
central_dir = PIPELINE_DIR / "CENTRAL"

c=Path(central_dir);p=c.parent
diagnostic=os.environ.get('PIPELINE_ALLOW_UNVERIFIED')=='1'
sources=collect_sources(p,diagnostic)
parts=[];quarantine=[]
for ds in DATASETS:
    d=p/'sub-dataset'/ds;verify_dataset(d)
    tables=read_tables(d/'output/harmonized');validate_tables(*tables,allow_quarantined=True)
    good=~quality_flags(tables[0],tables[2]).any(axis=1)
    quarantine.append({'source_dataset':ds,'source_rows':len(good),'eligible_rows':int(good.sum()),'quarantined_rows':int((~good).sum()),'quarantined_test_rows':int((~good & tables[2].source_split.astype(str).str.lower().isin(['test','kddtest'])).sum())})
    parts.append(tuple(t.loc[good].reset_index(drop=True) for t in tables))
f,l,m=[pd.concat([t[i] for t in parts],ignore_index=True) for i in range(3)]
l=map_labels(l,m);validate_tables(f,l,m)
unavailable={flag:int((~m[flag].astype(str).str.lower().isin(['true','1'])).sum()) for flag in ['has_service','has_connection_state','has_protocol']}
require(diagnostic or not any(unavailable.values()),f'Required attributes unavailable: {unavailable}. Supply measured fields or run diagnostic mode explicitly.')


## 5. แบ่ง train, validation และ test

จัดกลุ่มแถวที่มีฟีเจอร์หลักตรงกันก่อนสุ่ม validation ใช้ seed 42 และรักษาสมาชิก test จากต้นทาง


In [ ]:
# Split entire identical model-input groups together, without reading y.
groups=pd.util.hash_pandas_object(f[CORE],index=False)
official=m.source_split.astype(str).str.lower()
train_mask=official.isin(['train','kddtrain']);test_mask=official.isin(['test','kddtest'])
require((train_mask|test_mask).all(),'Unknown source split')
# Test features must not influence which training groups become validation.
unique=np.sort(groups[train_mask].unique());rng=np.random.default_rng(42)
val_groups=set(rng.permutation(unique)[:int(len(unique)*0.2)])
m['data_split']=np.where(test_mask,'test',np.where(groups.isin(val_groups),'validation','train'))
m['split_group']=groups.map(lambda v:f'core:{v:016x}')
m=normalize_weights(m)


## 6. บันทึกตารางและ split

เขียน features, labels และ metadata สำหรับ harmonized, train, train_full, validation และ test


In [ ]:
o=c/'output';h=o/'harmonized';h.mkdir(parents=True,exist_ok=True)
for name,frame in [('features',f),('labels',l),('metadata',m)]:frame.to_csv(h/(name+'.csv'),index=False)
for split in ['train','train_full','validation','test']:
    mask=m.data_split.isin(['train','validation']) if split=='train_full' else m.data_split.eq(split)
    dest=o/'splits'/split;dest.mkdir(parents=True,exist_ok=True)
    frames=(f.loc[mask],l.loc[mask],normalize_weights(m.loc[mask]))
    for name,frame in zip(['features','labels','metadata'],frames):frame.to_csv(dest/(name+'.csv'),index=False)


## 7. บันทึกรายงานตรวจสอบ

บันทึกจำนวนแถวที่ถูกคัดออก การกระจายของแหล่งข้อมูลและ label ผลตรวจ overlap และ manifest ของการสร้างข้อมูล


In [ ]:
reports=o/'reports';reports.mkdir(exist_ok=True)
pd.DataFrame(quarantine).to_csv(reports/'quarantine_summary.csv',index=False)
write_json(o/'data_sources.json',sources)
pd.crosstab([m.source_dataset,l.label_source_category],l.attack_category_id).to_csv(reports/'label_mapping.csv')
pd.crosstab(m.source_dataset,m.data_split).to_csv(reports/'source_splits.csv')
overlaps={}
for a,b in [('train','validation'),('train','test'),('validation','test')]:
    aa=set(groups[m.data_split.eq(a)]);bb=groups[m.data_split.eq(b)]
    overlaps[a+'_'+b]={'matching_rows_in_second':int(bb.isin(aa).sum()),'unique_vectors':len(aa.intersection(set(bb)))}
require(overlaps['train_validation']['matching_rows_in_second']==0,'Grouped validation split failed')
write_json(reports/'overlap.json',{'comparison':'six shared raw model inputs; identical features need not be identical events','overlaps':overlaps,'conflicting_label_vectors':int(pd.DataFrame({'g':groups,'y':l.attack_category_id}).groupby('g').y.nunique().gt(1).sum()),'supplied_test_membership_preserved_for_eligible_records':True,'quarantined_test_rows':sum(v['quarantined_test_rows'] for v in quarantine)})
write_json(o/'build_manifest.json',{'raw':{ds:raw_manifest(p/'sub-dataset'/ds) for ds in DATASETS},'code':code_signature(p),'rows':len(f),'split_policy':'seeded grouping of identical shared model inputs; original test rows preserved','unavailable_attributes':unavailable,'profile_scope':'diagnostic' if diagnostic else 'source_documented','class_taxonomy_note':'Project category mapping; MITRE fields are group annotations, not verified per-flow attribution'})
result = {'rows':len(f),'splits':m.data_split.value_counts().to_dict()}

print(result)
